# Short term data preparation for event study

# Tweets and financial markets

## Group Project

Cathy (Cui) Yu, Hefei Mao, Yichi Wang, Di Yang, Marc Hayes

### Imports

In [1]:
# Standard library
from datetime import datetime, date, time as dtime, timedelta
import glob
import os
import re
from pathlib import Path
from time import time
from typing import Dict, List, Tuple, Optional

# Third-party
import numpy as np
import pandas as pd
from pandas.errors import ParserError
import pytz

import warnings
# optional: quiet PerformanceWarning globally (use only if you still see stray warnings)
# warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

### Loading the twitter data

In [2]:
df_tweets = pd.read_csv("tweets_event_windows_final.csv")
df_tweets.head()

,new_id,id,text,date,event_minute,ticker,company_name,industry,event_time_et,in_trading_hours,adjusted_event_time_et,adjusted_event_time_utc,event_1h_pre_et,event_1h_end_et,event_5d_pre_et,event_5d_end_et,event_1h_pre,event_1h_end,event_5d_pre,event_5d_end
0,52,911287725847908352,Thank you to Doug Parker and American Airlines...,2017-09-22 17:54:59,2017-09-22 17:54:00+00:00,AAL,American Airlines Group,Airlines & Travel,2017-09-22 13:54:00-04:00,0,2017-09-22 13:54:00-04:00,2017-09-22 17:54:00+00:00,2017-09-22 12:54:00-04:00,2017-09-22 14:54:00-04:00,2017-09-15 09:30:00-04:00,2017-09-29 09:30:00-04:00,2017-09-22 16:54:00+00:00,2017-09-22 18:54:00+00:00,2017-09-15 13:30:00+00:00,2017-09-29 13:30:00+00:00
1,149,1184147319480041473,"Join me in Dallas, Texas this Thursday (Octobe...",2019-10-15 16:41:36,2019-10-15 16:41:00+00:00,AAL,American Airlines Group,Airlines & Travel,2019-10-15 12:41:00-04:00,0,2019-10-15 12:41:00-04:00,2019-10-15 16:41:00+00:00,2019-10-15 11:41:00-04:00,2019-10-15 13:41:00-04:00,2019-10-08 09:30:00-04:00,2019-10-22 09:30:00-04:00,2019-10-15 15:41:00+00:00,2019-10-15 17:41:00+00:00,2019-10-08 13:30:00+00:00,2019-10-22 13:30:00+00:00
2,148,1184631273454817280,"THANK YOU you Dallas, Texas. See you tomorrow ...",2019-10-17 00:44:39,2019-10-17 00:44:00+00:00,AAL,American Airlines Group,Airlines & Travel,2019-10-16 20:44:00-04:00,1,2019-10-17 09:30:00-04:00,2019-10-17 13:30:00+00:00,2019-10-17 08:30:00-04:00,2019-10-17 10:30:00-04:00,2019-10-10 09:30:00-04:00,2019-10-24 09:30:00-04:00,2019-10-17 12:30:00+00:00,2019-10-17 14:30:00+00:00,2019-10-10 13:30:00+00:00,2019-10-24 13:30:00+00:00
3,147,1184987864125321216,Just arrived at the American Airlines Center i...,2019-10-18 00:21:37,2019-10-18 00:21:00+00:00,AAL,American Airlines Group,Airlines & Travel,2019-10-17 20:21:00-04:00,1,2019-10-18 09:30:00-04:00,2019-10-18 13:30:00+00:00,2019-10-18 08:30:00-04:00,2019-10-18 10:30:00-04:00,2019-10-11 09:30:00-04:00,2019-10-25 09:30:00-04:00,2019-10-18 12:30:00+00:00,2019-10-18 14:30:00+00:00,2019-10-11 13:30:00+00:00,2019-10-25 13:30:00+00:00
4,24,700795170023825408,"I use both iPhone &amp, Samsung. If Apple does...",2016-02-19 21:32:43,2016-02-19 21:32:00+00:00,AAPL,Apple Inc.,Technology & Internet,2016-02-19 16:32:00-05:00,1,2016-02-22 09:30:00-05:00,2016-02-22 14:30:00+00:00,2016-02-22 08:30:00-05:00,2016-02-22 10:30:00-05:00,2016-02-12 09:30:00-05:00,2016-02-29 09:30:00-05:00,2016-02-22 13:30:00+00:00,2016-02-22 15:30:00+00:00,2016-02-12 14:30:00+00:00,2016-02-29 14:30:00+00:00


### Preprocessing twitter data

In [3]:
# For the short term analysis we focus on the minutely data, we decided to use Eastern Time

# Convert the relevant time columns from object type to pandas timestamp from UTC to ET
cols = ["event_time_et", "event_1h_pre_et", "event_1h_end_et"]

for col in cols:
    df_tweets[col] = (
        pd.to_datetime(df_tweets[col], utc=True, errors="coerce")
          .dt.tz_convert("America/New_York")
    )

# Convert "date" column to timestamp, but keep it in UTC -> "date_utc"
df_tweets["date_utc"] = pd.to_datetime(df_tweets["date"], utc=True, errors="coerce")

### Filtering the tweets to retain only relevant entries

In [4]:
df_tweets.columns

Index(['new_id', 'id', 'text', 'date', 'event_minute', 'ticker',
       'company_name', 'industry', 'event_time_et', 'in_trading_hours',
       'adjusted_event_time_et', 'adjusted_event_time_utc', 'event_1h_pre_et',
       'event_1h_end_et', 'event_5d_pre_et', 'event_5d_end_et', 'event_1h_pre',
       'event_1h_end', 'event_5d_pre', 'event_5d_end', 'date_utc'],
      dtype='object')

In [5]:
# Keep only the relevant columns for the long term analysis
df_tweets_filtered = df_tweets[[
    "id", 
    "text", 
    "date_utc",
    "event_time_et", 
    # "event_1h_pre_et", 
    # "event_1h_end_et", 
    "ticker", 
    "company_name", 
    "industry"
]]
df_tweets_filtered.head()

,id,text,date_utc,event_time_et,ticker,company_name,industry
0,911287725847908352,Thank you to Doug Parker and American Airlines...,2017-09-22 17:54:59+00:00,2017-09-22 13:54:00-04:00,AAL,American Airlines Group,Airlines & Travel
1,1184147319480041473,"Join me in Dallas, Texas this Thursday (Octobe...",2019-10-15 16:41:36+00:00,2019-10-15 12:41:00-04:00,AAL,American Airlines Group,Airlines & Travel
2,1184631273454817280,"THANK YOU you Dallas, Texas. See you tomorrow ...",2019-10-17 00:44:39+00:00,2019-10-16 20:44:00-04:00,AAL,American Airlines Group,Airlines & Travel
3,1184987864125321216,Just arrived at the American Airlines Center i...,2019-10-18 00:21:37+00:00,2019-10-17 20:21:00-04:00,AAL,American Airlines Group,Airlines & Travel
4,700795170023825408,"I use both iPhone &amp, Samsung. If Apple does...",2016-02-19 21:32:43+00:00,2016-02-19 16:32:00-05:00,AAPL,Apple Inc.,Technology & Internet


### Processing the stock data

In [6]:
# NOTE: Data preprocessing, only needs to run once and can then be commented out.
# For submission this cell is interrupted to save runtime.

# ---------- Config ----------
INPUT_DIR_CSV = "DATA EOD"
OUTPUT_DIR_HOURLY = "DATA HOURLY"
os.makedirs(OUTPUT_DIR_HOURLY, exist_ok=True)

# ET session config
ET = pytz.timezone("US/Eastern")
SESSION_OPEN_ET  = (9, 30)
SESSION_CLOSE_ET = (16, 0)

# ---------- Helper (from your minutely cell) ----------
def et_session_range_for_day(day: date):
    """
    Return (open_utc, close_utc) for the trading session on `day`,
    where open/close defined by SESSION_OPEN_ET / SESSION_CLOSE_ET in ET.
    """
    dt_open_et = datetime(day.year, day.month, day.day, SESSION_OPEN_ET[0], SESSION_OPEN_ET[1])
    dt_close_et = datetime(day.year, day.month, day.day, SESSION_CLOSE_ET[0], SESSION_CLOSE_ET[1])
    dt_open_et = ET.localize(dt_open_et)
    dt_close_et = ET.localize(dt_close_et)
    return dt_open_et.astimezone(pytz.UTC), dt_close_et.astimezone(pytz.UTC)

# ---------- Core: hourly aggregation normalized to 9:30 ET ----------
def process_file_hourly_normalized(input_path: str, output_path: str):
    """Read minute-level CSV (with timestamp), compute hourly bins anchored at 9:30 ET, output hourly aggregates."""
    print(f"Processing {input_path} ...")
    # robust read
    try:
        df = pd.read_csv(input_path)
    except Exception as e:
        print(f"  Read error, skipping: {e}")
        return

    # must have a timestamp column
    if "timestamp" not in df.columns and "datetime" not in df.columns:
        print(f"  Skipping {input_path}: no 'timestamp' or 'datetime' column found.")
        return

    # choose timestamp column and parse to tz-aware UTC
    ts_col = "timestamp" if "timestamp" in df.columns else "datetime"
    df[ts_col] = pd.to_datetime(df[ts_col], errors="coerce", utc=True)
    n_bad = df[ts_col].isna().sum()
    if n_bad:
        print(f"  Warning: {n_bad} rows have invalid timestamps and will be dropped.")
        df = df.dropna(subset=[ts_col])
    if df.empty:
        print("  No valid rows after timestamp parse; skipping.")
        return

    # canonicalize name
    df = df.rename(columns={ts_col: "timestamp"})
    # ensure tz-aware UTC
    if df["timestamp"].dt.tz is None:
        df["timestamp"] = df["timestamp"].dt.tz_localize("UTC")
    else:
        df["timestamp"] = df["timestamp"].dt.tz_convert("UTC")

    # keep only rows inside ET trading session for the corresponding day (filter out pre/post session)
    # convert to ET then compute date and session minutes
    df["timestamp_et"] = df["timestamp"].dt.tz_convert(ET)
    # compute day (ET-local) to determine session open/close
    df["et_date"] = df["timestamp_et"].dt.date

    # Build a boolean mask marking rows inside the session window for their day
    def in_session_row(row):
        # for each row, compute open/close in UTC and check timestamp between them
        open_utc, close_utc = et_session_range_for_day(row["et_date"])
        return (row["timestamp"] >= open_utc) and (row["timestamp"] <= close_utc)

    df = df[df.apply(in_session_row, axis=1)].copy()
    if df.empty:
        print("  No rows inside trading sessions -> skipping file.")
        return

    # Compute typical_price if possible (for VWAP)
    if {"high", "low", "close"}.issubset(df.columns):
        df["typical_price"] = (df["high"] + df["low"] + df["close"]) / 3.0
    else:
        # fallback to close only
        if "close" in df.columns:
            df["typical_price"] = df["close"]
        else:
            print("  Skipping: 'close' (or high/low/close) required.")
            return

    # Ensure volume present (if not, create zeroes)
    if "volume" not in df.columns:
        df["volume"] = 0

    # Compute bin index anchored at ET session open (9:30 ET)
    # For each row: minutes_since_open = (timestamp_utc - open_utc_for_day).total_seconds() // 60
    # bin_idx = floor(minutes_since_open / 60)
    def compute_bin_start_utc(row):
        open_utc, _ = et_session_range_for_day(row["et_date"])
        minutes_since_open = (row["timestamp"] - open_utc).total_seconds() / 60.0
        # guard: if negative (shouldn't happen due to earlier filter) set to 0
        if minutes_since_open < 0:
            minutes_since_open = 0.0
        bin_idx = int(np.floor(minutes_since_open / 60.0))
        bin_start_et = (open_utc.astimezone(ET) + timedelta(minutes=bin_idx * 60))
        # convert bin_start_et to UTC and return tz-aware UTC
        return bin_start_et.astimezone(pytz.UTC)

    # compute hour (bin start UTC) for each row
    df["hour"] = df.apply(compute_bin_start_utc, axis=1)

    # Now group by hour and compute aggregations
    # trades_in_hour = count of rows
    # price aggregates: mean of close/open/high/low where applicable; VWAP computed from typical_price * volume
    agg_dict = {}
    agg_dict["trades_in_hour"] = ("timestamp", "count")  # count of minute rows (or trade ticks if raw)
    # include price means if present
    if "open" in df.columns:
        agg_dict["open_mean"] = ("open", "mean")
    if "high" in df.columns:
        agg_dict["high_mean"] = ("high", "mean")
    if "low" in df.columns:
        agg_dict["low_mean"] = ("low", "mean")
    agg_dict["close_mean"] = ("close", "mean")
    agg_dict["typical_price_mean"] = ("typical_price", "mean")
    agg_dict["volume_sum"] = ("volume", "sum")

    # perform groupby on 'hour'
    df_hourly = df.groupby("hour").agg(**agg_dict).reset_index()

    # compute VWAP: sum(typical_price * volume) / sum(volume) per hour (avoid divide by zero)
    df["price_volume"] = df["typical_price"] * df["volume"]
    vwap = (
        df.groupby("hour")
        .agg(pv_sum=("price_volume", "sum"), vol_sum=("volume", "sum"))
        .reset_index()
    )
    vwap["hourly_vwap"] = vwap.apply(lambda r: (r["pv_sum"] / r["vol_sum"]) if (r["vol_sum"] and not np.isnan(r["vol_sum"])) else np.nan, axis=1)
    vwap = vwap[["hour", "hourly_vwap"]]
    df_hourly = df_hourly.merge(vwap, on="hour", how="left")

    # Sort by hour and compute returns (pct change of close_mean)
    df_hourly = df_hourly.sort_values("hour").reset_index(drop=True)
    df_hourly["hourly_return"] = df_hourly["close_mean"].pct_change()
    # log return (safe)
    df_hourly["hourly_log_return"] = np.log(df_hourly["close_mean"] / df_hourly["close_mean"].shift(1))

    # Save: 'hour' is tz-aware UTC. Keep it as ISO string or timestamp; we keep ISO with tz for clarity.
    # Reset index to make 'hour' a column (already is)
    df_hourly.to_csv(output_path, index=False)
    print(f"  Saved hourly normalized file to {output_path} (rows: {len(df_hourly)})")


# ---------- Runner: iterate CSVs ----------
def build_hourly_all_files(input_dir=INPUT_DIR_CSV, output_dir=OUTPUT_DIR_HOURLY):
    csv_files = [
        os.path.basename(p)
        for p in glob.glob(os.path.join(input_dir, "*.csv"))
        if os.path.basename(p).lower() != "tickers.csv"
    ]

    if not csv_files:
        print("No CSV files found in INPUT_DIR.")
        return

    for file in csv_files:
        input_path = os.path.join(input_dir, file)
        # create output name similar to before but with '_hourly' suffix
        out_name = f"{os.path.splitext(file)[0]}_hourly.csv"
        out_path = os.path.join(output_dir, out_name)
        process_file_hourly_normalized(input_path, out_path)

# Example usage:
build_hourly_all_files()

Processing DATA EOD\AAL.csv ...
  Saved hourly normalized file to DATA HOURLY\AAL_hourly.csv (rows: 14404)
Processing DATA EOD\AAPL.csv ...
  Saved hourly normalized file to DATA HOURLY\AAPL_hourly.csv (rows: 23075)
Processing DATA EOD\AMD.csv ...
  Saved hourly normalized file to DATA HOURLY\AMD_hourly.csv (rows: 23072)
Processing DATA EOD\AMZN.csv ...
  Saved hourly normalized file to DATA HOURLY\AMZN_hourly.csv (rows: 23071)
Processing DATA EOD\APD.csv ...
  Saved hourly normalized file to DATA HOURLY\APD_hourly.csv (rows: 23073)
Processing DATA EOD\APTV.csv ...


KeyboardInterrupt: 

In [7]:
df_amd_hourly = pd.read_csv("DATA HOURLY/AMD_hourly.csv")
df_amd_hourly

,hour,trades_in_hour,open_mean,high_mean,low_mean,close_mean,typical_price_mean,volume_sum,hourly_vwap,hourly_return,hourly_log_return
0,2009-01-02 14:30:00+00:00,60,2.204177,2.208000,2.197192,2.204223,2.203138,1381618.0,2.206952,NaN,NaN
1,2009-01-02 15:30:00+00:00,60,2.266790,2.272000,2.259413,2.267528,2.266314,2278563.0,2.264744,0.028720,0.028315
2,2009-01-02 16:30:00+00:00,60,2.311782,2.316597,2.305082,2.312962,2.311547,2040396.0,2.311604,0.020037,0.019838
3,2009-01-02 17:30:00+00:00,60,2.351673,2.357500,2.346663,2.353445,2.352536,1610116.0,2.357058,0.017503,0.017351
4,2009-01-02 18:30:00+00:00,60,2.383267,2.388333,2.377380,2.384333,2.383349,1664141.0,2.381657,0.013125,0.013039
...,...,...,...,...,...,...,...,...,...,...,...
23067,2022-02-18 16:30:00+00:00,60,111.701332,111.846898,111.597887,111.720553,111.721779,12156305.0,111.731417,0.006703,0.006680
23068,2022-02-18 17:30:00+00:00,60,111.272737,111.370017,111.162237,111.261145,111.264466,10184992.0,111.292709,-0.004112,-0.004121
23069,2022-02-18 18:30:00+00:00,60,112.804033,112.954410,112.731390,112.864185,112.849995,15551338.0,113.030307,0.014408,0.014305
23070,2022-02-18 19:30:00+00:00,60,114.803257,114.941495,114.653630,114.787287,114.794137,19343590.0,114.842258,0.017039,0.016896


In [8]:
# ---------- CONFIG (adjust as needed) ----------
HOURLY_DIR = "DATA HOURLY"                                # where <TICKER>_hourly.csv live
OUTPUT_DIR_COMPANY_HOURLY = "HOURLY EVENT STUDY PANELS BY COMPANY"
os.makedirs(OUTPUT_DIR_COMPANY_HOURLY, exist_ok=True)

# ET session (DST-aware)
ET = pytz.timezone("US/Eastern")
SESSION_OPEN_ET = (9, 30)    # session start (9:30 ET)
SESSION_CLOSE_ET = (16, 0)   # session end (16:00 ET)

# default hourly window: 5 hours before, 2 after
N_BEFORE = 5
N_AFTER = 2

# columns to extract from per-ticker hourly files (only returns as requested)
TICKER_COLUMNS = [
    "trades_in_hour",
    "open_mean",
    "high_mean",
    "low_mean",
    "close_mean",
    "typical_price_mean",
    "volume_sum",
    "hourly_vwap",
    "hourly_return", 
    "hourly_log_return",
]
# ------------------------------------------------

# --- Holiday/trading-day helpers ---
_USE_MCAL = False
try:
    import pandas_market_calendars as mcal
    _USE_MCAL = True
except Exception:
    _USE_MCAL = False

def trading_days_between(start_date: date, end_date: date) -> List[date]:
    """
    Return a list of trading dates between start_date and end_date inclusive.
    Prefer NYSE calendar (pandas_market_calendars) if available; otherwise
    fall back to business days minus US federal holidays (conservative).
    """
    if start_date > end_date:
        return []

    if _USE_MCAL:
        # Use NYSE schedule
        nyse = mcal.get_calendar("NYSE")
        schedule = nyse.schedule(start_date=start_date.isoformat(), end_date=end_date.isoformat())
        # schedule.index are trading days in pandas Timestamp (date at midnight ET)
        # convert to python date
        return [pd.Timestamp(d).date() for d in schedule.index]
    else:
        # Fallback: business days excluding US federal holidays (conservative)
        # This is not identical to NYSE holidays but is a reasonable fallback.
        from pandas.tseries.holiday import USFederalHolidayCalendar
        bdays = pd.bdate_range(start=start_date, end=end_date).date
        cal = USFederalHolidayCalendar()
        holidays = cal.holidays(start=start_date, end=end_date).date
        trading_days = [d for d in bdays if d not in set(holidays)]
        # warn user once that fallback is less accurate than NYSE calendar
        print("Warning: pandas_market_calendars not available - using US federal holidays as fallback (may differ from NYSE holidays).")
        return trading_days

# --- session hours builder ---
__SESSION_HOURS_CACHE = {}

def et_session_range_for_day(day: date) -> Tuple[datetime, datetime]:
    """Return UTC-aware datetimes for session open/close on a given day."""
    dt_open_et = datetime(day.year, day.month, day.day, SESSION_OPEN_ET[0], SESSION_OPEN_ET[1])
    dt_close_et = datetime(day.year, day.month, day.day, SESSION_CLOSE_ET[0], SESSION_CLOSE_ET[1])
    dt_open_et = ET.localize(dt_open_et)
    dt_close_et = ET.localize(dt_close_et)
    return dt_open_et.astimezone(pytz.UTC), dt_close_et.astimezone(pytz.UTC)

def build_session_hours(start_date: date, end_date: date, force_rebuild: bool = False) -> pd.DatetimeIndex:
    """
    Build UTC DatetimeIndex of session-hour anchors for each trading day between start_date and end_date inclusive.
    Anchors: 9:30 ET, 10:30 ET, ..., 15:30 ET (converted to UTC).
    Uses trading day list that excludes weekends and market holidays (if available).
    """
    key = (start_date.isoformat(), end_date.isoformat())
    if (not force_rebuild) and (key in __SESSION_HOURS_CACHE):
        return __SESSION_HOURS_CACHE[key]

    tdays = trading_days_between(start_date, end_date)
    all_hours = []
    for d in tdays:
        open_utc, close_utc = et_session_range_for_day(d)
        # produce anchors starting at open_utc, step 60min, stop at last anchor <= close_utc - 30min
        # pd.date_range with freq 60min starting at open_utc will produce anchors like 9:30,10:30,... up to close if exact
        hrs = pd.date_range(start=open_utc, end=close_utc - pd.Timedelta(minutes=30), freq="60min", tz=pytz.UTC)
        # typical anchors: 9:30,10:30,...,15:30 (given session_close 16:00 -> last anchor 15:30)
        all_hours.append(hrs)

    if not all_hours:
        idx = pd.DatetimeIndex([], tz=pytz.UTC)
    else:
        idx = all_hours[0]
        for hrs in all_hours[1:]:
            idx = idx.union(hrs)
        idx = pd.DatetimeIndex(sorted(idx))
        if idx.tz is None:
            idx = idx.tz_localize(pytz.UTC)

    __SESSION_HOURS_CACHE[key] = idx
    return idx

def _find_centered_window_hours(session_index: pd.DatetimeIndex, event_ts_utc: pd.Timestamp,
                                n_before: int = N_BEFORE, n_after: int = N_AFTER) -> pd.DatetimeIndex:
    """
    Given session anchors and event timestamp, choose anchors centered around the
    first anchor >= event_ts (searchsorted side='left'); this means tweets after session close
    will map to next session open anchor; tweets before session open will map to that day's first anchor.
    Return an index consisting of n_before anchors before the selected anchor (if available)
    and n_after anchors after it (if available), in order.
    """
    if len(session_index) == 0:
        return pd.DatetimeIndex([], tz=pytz.UTC)
    event_ts_utc = pd.to_datetime(event_ts_utc, utc=True)
    pos = session_index.searchsorted(event_ts_utc, side="left")
    # If pos == len(session_index), searchsorted placed event after last anchor in index -> pos == len -> we can't index at pos
    # but we still want the next anchor (which doesn't exist in current index) -> we'll get NaTs for those slots.
    desired_positions = [pos - n_before + i for i in range(n_before + n_after + 1)]
    desired_timestamps = []
    for p in desired_positions:
        if 0 <= p < len(session_index):
            desired_timestamps.append(session_index[p])
        else:
            desired_timestamps.append(pd.NaT)
    return pd.DatetimeIndex(desired_timestamps)

# --- hourly file IO / discovery ---
def load_hourly_df(ticker: str, hourly_dir: str = HOURLY_DIR) -> Optional[pd.DataFrame]:
    """
    Load hourly CSV for ticker. Accepts files like "<TICKER>_hourly.csv" or "<TICKER>_hourly_v2.csv".
    Ensures UTC tz-aware DatetimeIndex.
    """
    path = os.path.join(hourly_dir, f"{ticker}_hourly.csv")
    if not os.path.exists(path):
        found = glob.glob(os.path.join(hourly_dir, f"{ticker}*_hourly*.csv"))
        if found:
            path = found[0]
        else:
            return None
    try:
        df = pd.read_csv(path)
    except Exception as e:
        print(f"  Error reading {path}: {e}")
        return None

    # Acceptable time cols: 'timestamp', 'hour', 'datetime'
    time_col = None
    for c in ("timestamp", "hour", "datetime"):
        if c in df.columns:
            time_col = c
            break
    if time_col is None:
        # try to infer a datetime-like column
        for c in df.columns:
            try:
                parsed = pd.to_datetime(df[c], errors="coerce")
                if parsed.notna().sum() > 0.5 * len(df):
                    time_col = c
                    break
            except Exception:
                continue
    if time_col is None:
        print(f"  File {path} has no recognized time column -> skipping.")
        return None

    df[time_col] = pd.to_datetime(df[time_col], utc=True, errors="coerce")
    if df[time_col].isna().any():
        df = df.dropna(subset=[time_col])
    if df.empty:
        return None

    df = df.set_index(time_col).sort_index()
    # ensure tz-aware UTC index
    if df.index.tz is None:
        df.index = df.index.tz_localize(pytz.UTC)
    else:
        df.index = df.index.tz_convert(pytz.UTC)

    return df

def _collect_ticker_list_hourly(hourly_dir: str = HOURLY_DIR, tickers: Optional[List[str]] = None) -> List[str]:
    """
    Discover tickers by scanning files like "*_hourly*.csv" in hourly_dir.
    If tickers list provided, return it.
    """
    if tickers:
        return tickers
    files = glob.glob(os.path.join(hourly_dir, "*_hourly*.csv"))
    tickers_found = []
    for p in files:
        bn = os.path.basename(p)
        if "_hourly" in bn:
            ticker = bn.split("_hourly")[0]
            tickers_found.append(ticker)
    tickers_found = sorted(set(tickers_found))
    return tickers_found

# --- main builder ---
def build_hourly_event_panel(tweet_id: int,
                             df_tweets: pd.DataFrame,
                             event_ts_col: str = "event_ts_utc",
                             hourly_dir: str = HOURLY_DIR,
                             tickers: Optional[List[str]] = None,
                             session_start_date: Optional[date] = None,
                             session_end_date: Optional[date] = None,
                             n_before: int = N_BEFORE,
                             n_after: int = N_AFTER,
                             save_csv: bool = True,
                             output_dir: str = OUTPUT_DIR_COMPANY_HOURLY) -> Tuple[pd.DataFrame, dict]:
    """
    Build an hourly event panel for a single tweet.
    Output columns: 't', 'timestamp' (UTC anchor), and for each ticker:
      <TICKER>_hourly_return, <TICKER>_hourly_log_return
    """
    # 1) find tweet timestamp
    sel = df_tweets.loc[df_tweets["id"] == tweet_id, event_ts_col]
    if len(sel) == 0:
        raise KeyError(f"Tweet id {tweet_id} not found in df_tweets (expected column 'id').")
    event_ts = pd.to_datetime(sel.iloc[0], utc=True, errors="coerce")
    if pd.isna(event_ts):
        raise ValueError(f"Event timestamp for tweet {tweet_id} could not be parsed.")

    # 2) session index range (expand +/- 7 days by default)
    if session_start_date is None or session_end_date is None:
        ev_date = event_ts.date()
        session_start_date = ev_date - timedelta(days=7)
        session_end_date = ev_date + timedelta(days=7)

    session_idx = build_session_hours(session_start_date, session_end_date)
    if len(session_idx) == 0:
        raise RuntimeError("Built empty session index for provided date range. Check dates/holiday calendar.")

    # 3) compute desired (centered) timestamps using the searchsorted side='left' behavior:
    # event maps to first anchor >= event_ts (so tweets after close map to next open anchor).
    desired_idx = _find_centered_window_hours(session_idx, event_ts, n_before=n_before, n_after=n_after)
    # Build panel skeleton with t and timestamp
    panel_df = pd.DataFrame({"t": list(range(-n_before, n_after + 1)), "timestamp": list(desired_idx)})

    # 4) collect tickers and extract returns
    ticker_list = _collect_ticker_list_hourly(hourly_dir, tickers)
    processed = []
    skipped = []

    for ticker in ticker_list:
        df_tick = load_hourly_df(ticker, hourly_dir=hourly_dir)
        if df_tick is None or df_tick.empty:
            skipped.append((ticker, "missing_or_empty"))
            empty_cols = {f"{ticker}_{col}": [np.nan] * len(panel_df) for col in TICKER_COLUMNS}
            if empty_cols:
                panel_df = pd.concat([panel_df, pd.DataFrame(empty_cols, index=panel_df.index)], axis=1)
            continue

        vals = {col: [] for col in TICKER_COLUMNS}
        for ts in panel_df["timestamp"]:
            if pd.isna(ts):
                for col in TICKER_COLUMNS:
                    vals[col].append(np.nan)
                continue
            # exact-match on hourly anchor index
            if ts in df_tick.index:
                row = df_tick.loc[ts]
                if isinstance(row, pd.DataFrame):
                    row = row.iloc[0]
                for col in TICKER_COLUMNS:
                    vals[col].append(row[col] if col in row.index else np.nan)
            else:
                # not found -> NaN
                for col in TICKER_COLUMNS:
                    vals[col].append(np.nan)

        ticker_df = pd.DataFrame(vals, index=panel_df.index)
        ticker_df = ticker_df.rename(columns={c: f"{ticker}_{c}" for c in ticker_df.columns})
        panel_df = pd.concat([panel_df, ticker_df], axis=1)
        processed.append(ticker)

    panel_df = panel_df.copy()  # defragment

    info = {
        "tweet_id": tweet_id,
        "event_ts": event_ts.isoformat(),
        "session_start_date": session_start_date.isoformat(),
        "session_end_date": session_end_date.isoformat(),
        "n_tickers_processed": len(processed),
        "n_tickers_skipped": len(skipped),
        "skipped": skipped,
        "n_rows": len(panel_df),
        "t_range": (-n_before, n_after)
    }

    # 5) save CSV if requested
    if save_csv:
        os.makedirs(output_dir, exist_ok=True)
        out_path = os.path.join(output_dir, f"{tweet_id}_hourly_panel.csv")
        df_to_save = panel_df.copy()
        # convert timestamps to iso strings for CSV portability
        df_to_save["timestamp"] = df_to_save["timestamp"].apply(lambda x: x.isoformat() if pd.notna(x) else "")
        df_to_save.to_csv(out_path, index=False)
        info["saved_to"] = out_path
        print(f"Saved hourly panel for tweet {tweet_id} to {out_path}")

    return panel_df, info

In [9]:
# Example: build hourly panel for the first filtered tweet (no CSV save)
panel, info = build_hourly_event_panel(
    tweet_id = df_tweets_filtered["id"].iloc[0],
    df_tweets = df_tweets_filtered,
    event_ts_col = "date_utc",     # use same column name you used for minute version
    hourly_dir = "DATA HOURLY",    # where your <TICKER>_hourly.csv files live
    session_start_date = date(2015, 12, 1),
    session_end_date   = date(2020, 1, 31),
    n_before = 5,   # 5 hours pre-event
    n_after  = 2,   # 2 hours post-event
    save_csv = False,
    output_dir = None
)

# show the built panel (t, timestamp, and per-ticker hourly_return/hourly_log_return cols)
panel

,t,timestamp,AAL_trades_in_hour,AAL_open_mean,AAL_high_mean,AAL_low_mean,AAL_close_mean,AAL_typical_price_mean,AAL_volume_sum,AAL_hourly_vwap,...,XOM_trades_in_hour,XOM_open_mean,XOM_high_mean,XOM_low_mean,XOM_close_mean,XOM_typical_price_mean,XOM_volume_sum,XOM_hourly_vwap,XOM_hourly_return,XOM_hourly_log_return
0,-5,2017-09-22 13:30:00+00:00,60.0,46.888968,46.932508,46.849022,46.896880,46.892803,1355841.0,46.857484,...,60.0,79.983663,80.011592,79.963283,79.990618,79.988498,1331112.0,79.970239,0.001969,0.001968
1,-4,2017-09-22 14:30:00+00:00,60.0,47.066183,47.085833,47.042502,47.065185,47.064507,694411.0,47.084277,...,60.0,80.208218,80.222833,80.194685,80.208217,80.208578,1172548.0,80.211255,0.002720,0.002717
2,-3,2017-09-22 15:30:00+00:00,60.0,47.095662,47.114357,47.082962,47.099783,47.099034,579590.0,47.120853,...,60.0,80.204070,80.213148,80.194425,80.203625,80.203733,1102610.0,80.209343,-0.000057,-0.000057
3,-2,2017-09-22 16:30:00+00:00,60.0,47.260292,47.272833,47.245857,47.258382,47.259024,532821.0,47.283102,...,60.0,79.856143,79.868167,79.840047,79.850930,79.853048,1360125.0,79.860339,-0.004397,-0.004407
4,-1,2017-09-22 17:30:00+00:00,60.0,47.236848,47.250423,47.222763,47.237217,47.236801,504397.0,47.231950,...,60.0,79.783118,79.793073,79.772302,79.782237,79.782537,871714.0,79.778295,-0.000860,-0.000861
5,0,2017-09-22 18:30:00+00:00,60.0,47.207355,47.219317,47.196495,47.207280,47.207697,498611.0,47.207965,...,60.0,79.813797,79.823887,79.805555,79.815055,79.814832,803296.0,79.816209,0.000411,0.000411
6,1,2017-09-22 19:30:00+00:00,31.0,47.122577,47.136771,47.104713,47.119355,47.120280,783650.0,47.103906,...,31.0,79.880777,79.892448,79.868355,79.883155,79.881319,1788404.0,79.911384,0.000853,0.000853
7,2,2017-09-25 13:30:00+00:00,60.0,47.591152,47.646088,47.555093,47.605548,47.602243,1476244.0,47.598592,...,60.0,80.363197,80.390708,80.344692,80.371683,80.369028,1747156.0,80.313996,0.006116,0.006097


In [10]:
# --- CONFIG ---
HOURLY_DIR = "DATA HOURLY"                                  # where <TICKER>_hourly.csv live
OUTPUT_DIR_COMPANY_HOURLY = "HOURLY EVENT STUDY PANELS BY COMPANY"
os.makedirs(OUTPUT_DIR_COMPANY_HOURLY, exist_ok=True)

# ET session window details are taken from your hourly helper (9:30-16:00 ET normalized to UTC)
SESSION_START_DATE = date(2015, 12, 1)
SESSION_END_DATE   = date(2020, 1, 31)

N_BEFORE_HOURS = 5   # 5 hours pre-event
N_AFTER_HOURS  = 2   # 2 hours post-event
# ------------------

processed = 0
skipped = []

for tweet_id in df_tweets_filtered["id"].values:
    print(f"Processing tweet id: {tweet_id} ...")
    try:
        panel, info = build_hourly_event_panel(
            tweet_id = int(tweet_id),             # ensure numeric id if needed
            df_tweets = df_tweets_filtered,
            event_ts_col = "date_utc",            # change if your event column differs
            hourly_dir = HOURLY_DIR,
            session_start_date = SESSION_START_DATE,
            session_end_date   = SESSION_END_DATE,
            n_before = N_BEFORE_HOURS,
            n_after  = N_AFTER_HOURS,
            save_csv = True,
            output_dir = OUTPUT_DIR_COMPANY_HOURLY
        )
        processed += 1
        print("  done —", info.get("saved_to", "saved (no path returned)"))
    except KeyboardInterrupt:
        raise
    except Exception as e:
        print(f"  Skipped (error): {e}")
        skipped.append((tweet_id, str(e)))
    print()

print(f"Finished. Processed: {processed}. Skipped: {len(skipped)}.")
if skipped:
    print("Skipped items (tweet_id, reason):")
    for t, reason in skipped:
        print(" ", t, reason)

Processing tweet id: 911287725847908352 ...
Saved hourly panel for tweet 911287725847908352 to HOURLY EVENT STUDY PANELS BY COMPANY\911287725847908352_hourly_panel.csv
  done — HOURLY EVENT STUDY PANELS BY COMPANY\911287725847908352_hourly_panel.csv

Processing tweet id: 1184147319480041473 ...
Saved hourly panel for tweet 1184147319480041473 to HOURLY EVENT STUDY PANELS BY COMPANY\1184147319480041473_hourly_panel.csv
  done — HOURLY EVENT STUDY PANELS BY COMPANY\1184147319480041473_hourly_panel.csv

Processing tweet id: 1184631273454817280 ...
Saved hourly panel for tweet 1184631273454817280 to HOURLY EVENT STUDY PANELS BY COMPANY\1184631273454817280_hourly_panel.csv
  done — HOURLY EVENT STUDY PANELS BY COMPANY\1184631273454817280_hourly_panel.csv

Processing tweet id: 1184987864125321216 ...
Saved hourly panel for tweet 1184987864125321216 to HOURLY EVENT STUDY PANELS BY COMPANY\1184987864125321216_hourly_panel.csv
  done — HOURLY EVENT STUDY PANELS BY COMPANY\1184987864125321216_ho

KeyboardInterrupt: 

In [ ]:
# NOTE: Above code was interrupted in submission notebook to save runtime. Files can be found in corresponding directory.

### Per Industry

In [11]:
def build_industry_panel_from_company_panel_returns_only(
    panel_df: pd.DataFrame,
    mapping_path: str = None,
    mapping_df: pd.DataFrame = None,
    industry_col_name: str = "Industry",
    ticker_col_name: str = "Ticker",
    how_return: str = "equal",   # "equal" or "volume"
    return_suffixes: Optional[List[str]] = None
) -> pd.DataFrame:
    """
    Aggregate company-level panel into an industry-level panel producing **returns only**.

    Parameters
    ----------
    panel_df : pd.DataFrame
        Company-level event panel (minute/hour/day). Should contain columns like
        "<TICKER>_hourly_return" or "<TICKER>_return" etc. Also may contain 't' and 'timestamp'.
    mapping_path : str, optional
        Path to csv/xlsx mapping file with columns [ticker_col_name, industry_col_name].
    mapping_df : pd.DataFrame, optional
        DataFrame mapping tickers -> industries (alternative to mapping_path).
    industry_col_name : str
        Column name in mapping with industry/category.
    ticker_col_name : str
        Column name in mapping with the ticker symbol.
    how_return : str
        "equal" (equal-weighted) or "volume" (volume-weighted; requires volume columns present).
    return_suffixes : list of str, optional
        List of suffix patterns (strings) to consider as return columns, in preferred order.
        Default: ["_hourly_return", "_minutely_return", "_daily_return", "_return", "_log_return"]

    Returns
    -------
    out_df : pd.DataFrame
        DataFrame preserving 't' and 'timestamp' (if present) and containing one column per industry:
        "<SafeIndustry>_return" (safe: non-alnum replaced with underscore).
    """
    if mapping_df is None and mapping_path is None:
        raise ValueError("Provide mapping_path or mapping_df (ticker -> industry).")

    # Load mapping if necessary
    if mapping_df is None:
        if mapping_path.lower().endswith((".xls", ".xlsx")):
            mapping_df = pd.read_excel(mapping_path)
        else:
            mapping_df = pd.read_csv(mapping_path)

    # normalize ticker strings
    if ticker_col_name not in mapping_df.columns or industry_col_name not in mapping_df.columns:
        raise KeyError(f"Mapping must contain columns: {ticker_col_name}, {industry_col_name}")

    mapping_df = mapping_df.copy()
    mapping_df[ticker_col_name] = mapping_df[ticker_col_name].astype(str).str.strip()

    # default suffixes (preferred order)
    if return_suffixes is None:
        return_suffixes = ["_hourly_return", "_minutely_return", "_daily_return", "_return", "_log_return", "_log_ret"]

    # 1) detect tickers in panel_df by scanning columns for return-like tokens
    cols = list(panel_df.columns)
    detected_tickers = []
    # Prefer exact suffix matches in provided order
    for suf in return_suffixes:
        for c in cols:
            if c.endswith(suf):
                ticker = c[: -len(suf)]
                if ticker not in detected_tickers:
                    detected_tickers.append(ticker)

    # fallback: any column that contains 'return' -> prefix before first '_' or '.' or whitespace
    if not detected_tickers:
        for c in cols:
            if "return" in c.lower():
                parts = re.split(r'[_\.\s]', c)
                if parts:
                    t = parts[0]
                    if t not in detected_tickers:
                        detected_tickers.append(t)

    detected_tickers = sorted(set(detected_tickers))
    if not detected_tickers:
        raise ValueError("No return-like columns found in panel_df. Ensure panel has columns like '<TICKER>_return'.")

    # 2) subset mapping to tickers present in the panel (ignore others)
    mapping_sub = mapping_df[mapping_df[ticker_col_name].isin(detected_tickers)].copy()
    if mapping_sub.empty:
        # preserve skeleton with t and timestamp if present, but no industry columns
        out_df = pd.DataFrame(index=panel_df.index)
        for c in ("t", "timestamp"):
            if c in panel_df.columns:
                out_df[c] = panel_df[c].values
        return out_df

    mapped_tickers = set(mapping_sub[ticker_col_name].tolist())

    # 3) build industry groups from mapping_sub only
    industry_groups = mapping_sub.groupby(industry_col_name)[ticker_col_name].apply(list).to_dict()

    # 4) prepare output skeleton, preserve 't' and 'timestamp'
    out_df = pd.DataFrame(index=panel_df.index)
    for c in ("t", "timestamp"):
        if c in panel_df.columns:
            out_df[c] = panel_df[c].values

    # helper: find return columns and optional volume column for a ticker
    def find_return_and_volume_cols(ticker):
        ret_cols = []
        log_cols = []
        vol_cols = []
        # prioritized matches by suffix list
        for suf in return_suffixes:
            cand = f"{ticker}{suf}"
            if cand in panel_df.columns:
                # classify as log if 'log' in suffix or in column name
                if "log" in suf or "log" in cand.lower():
                    log_cols.append(cand)
                else:
                    ret_cols.append(cand)
        # also accept any column that starts with ticker and contains 'return' if none found yet
        if not ret_cols and not log_cols:
            matches = [c for c in panel_df.columns if c.startswith(f"{ticker}") and "return" in c.lower()]
            for c in matches:
                if "log" in c.lower():
                    log_cols.append(c)
                else:
                    ret_cols.append(c)
        # volume variants
        for v_suf in ("_daily_volume", "_volume", "_vol", "_hourly_volume", "_minutely_volume"):
            vcol = f"{ticker}{v_suf}"
            if vcol in panel_df.columns:
                vol_cols.append(vcol)
        # deduplicate while preserving order
        def dedup(seq):
            seen = set(); out=[]
            for x in seq:
                if x not in seen:
                    seen.add(x); out.append(x)
            return out
        return dedup(ret_cols), dedup(log_cols), dedup(vol_cols)

    # 5) compute industry returns only for industries that actually have at least 1 mapped ticker
    for industry, tickers in industry_groups.items():
        # only keep tickers actually present in mapping_sub
        tickers_present = [t for t in tickers if t in mapped_tickers]
        if not tickers_present:
            continue

        # collect columns
        ret_cols = []
        log_cols = []
        vol_cols = []
        for t in tickers_present:
            rcs, lcs, vcs = find_return_and_volume_cols(t)
            ret_cols += rcs
            log_cols += lcs
            vol_cols += vcs

        # dedup lists preserving order
        def dedup_keep_order(seq):
            seen = set(); out=[]
            for x in seq:
                if x not in seen:
                    seen.add(x); out.append(x)
            return out
        ret_cols = dedup_keep_order(ret_cols)
        log_cols = dedup_keep_order(log_cols)
        vol_cols = dedup_keep_order(vol_cols)

        # skip if no returns at all for this industry
        if not ret_cols and not log_cols:
            continue

        rets_df = panel_df[ret_cols].astype(float) if ret_cols else pd.DataFrame(index=panel_df.index)
        logs_df = panel_df[log_cols].astype(float) if log_cols else pd.DataFrame(index=panel_df.index)
        vols_df = panel_df[vol_cols].astype(float) if vol_cols else pd.DataFrame(index=panel_df.index)

        # compute industry return
        if how_return == "equal":
            if not rets_df.empty:
                industry_return = rets_df.mean(axis=1, skipna=True)
            elif not logs_df.empty:
                industry_return = logs_df.mean(axis=1, skipna=True)
            else:
                industry_return = pd.Series([np.nan]*len(panel_df), index=panel_df.index)
        elif how_return == "volume":
            # volume-weighted; require both volumes and returns
            if vols_df.empty or rets_df.empty:
                industry_return = pd.Series([np.nan]*len(panel_df), index=panel_df.index)
            else:
                vol_sum = vols_df.fillna(0).sum(axis=1)
                ret_num = (rets_df.fillna(0) * vols_df.fillna(0)).sum(axis=1)
                industry_return = ret_num.div(vol_sum).where(vol_sum != 0, np.nan)
        else:
            raise ValueError("how_return must be 'equal' or 'volume'")

        # only add if we can compute (some non-NaN entries)
        if industry_return.notna().any():
            safe_ind = re.sub(r'[^0-9A-Za-z_]', '_', str(industry))
            out_df[f"{safe_ind}_return"] = industry_return.values

    return out_df

In [12]:
# Single validation run
df_ind = build_industry_panel_from_company_panel_returns_only(
    panel_df=panel,
    mapping_path="tickers.xlsx",
    industry_col_name="Industry",
    ticker_col_name="Ticker",
    how_return="equal",
    # optional: customize if you have nonstandard column names
    return_suffixes=["_hourly_return", "_hourly_log_return", "_hourly_ret", "_return", "_log_return"]
)

df_ind

,t,timestamp,Airlines___Travel_return,Automobiles___Auto_Components_return,Chemicals__Materials_sector__return,Construction___Infrastructure_return,Defense___Aerospace_return,Financials__Banks__return,News___Publishing_return,Oil___Natural_Gas_return,Semiconductors__Information_Technology_sector__return,Technology___Internet_return
0,-5,2017-08-15 15:30:00,0.000667,0.000145,0.000443,-0.002695,0.000517,-0.001527,-0.000804,-0.000193,0.001526,0.000491
1,-4,2017-08-15 16:30:00,-0.000618,-0.000996,-0.000171,0.000004,-0.001663,-0.000273,-0.003665,0.000630,0.000684,0.000757
2,-3,2017-08-15 17:30:00,0.000722,-0.000936,-0.000153,0.002390,-0.000735,0.000211,-0.002843,0.003697,0.000503,-0.000853
3,-2,2017-08-15 18:30:00,-0.000069,0.000531,0.000884,0.001250,0.001361,0.001280,-0.000791,0.000907,0.001307,0.000541
4,-1,2017-08-15 19:30:00,0.000083,0.000262,0.001481,-0.000174,0.000819,-0.000798,-0.003009,0.000712,0.000042,0.000248
5,0,2017-08-16 13:30:00,0.002336,0.000190,0.003282,-0.002199,0.000789,0.000220,-0.000625,-0.001750,-0.003993,-0.002157
6,1,2017-08-16 14:30:00,0.000734,0.002464,0.002186,-0.004210,-0.000099,-0.003128,-0.002956,-0.003646,-0.000025,0.002633
7,2,2017-08-16 15:30:00,0.002100,0.000426,0.001807,0.000182,0.000372,-0.000247,0.004365,-0.003890,-0.000789,0.002059


In [13]:
INPUT_DIR_COMPANY = "HOURLY EVENT STUDY PANELS BY COMPANY"
OUTPUT_DIR_INDUSTRY = "HOURLY EVENT STUDY PANELS BY INDUSTRY"
MAPPING_PATH = "tickers.xlsx"
HOW_RETURN = "equal"  # or "volume"

os.makedirs(OUTPUT_DIR_INDUSTRY, exist_ok=True)
panel_files = sorted(glob.glob(os.path.join(INPUT_DIR_COMPANY, "*.csv")))

print(f"About to process {len(panel_files)} CSV files (company level -> industry aggregate)")

for file_path in panel_files:
    fname = os.path.basename(file_path)
    print(f"Processing {fname} ...")
    try:
        panel_df = pd.read_csv(file_path)
    except Exception as e:
        print(f"  Skipping (read error): {e}")
        continue

    try:
        df_ind = build_industry_panel_from_company_panel_returns_only(
            panel_df=panel_df,
            mapping_path=MAPPING_PATH,
            industry_col_name="Industry",
            ticker_col_name="Ticker",
            how_return=HOW_RETURN
        )
    except Exception as e:
        print(f"  Skipping (error during aggregation): {e}")
        continue

    # if df_ind contains only t/timestamp (no industry return columns) you may want to skip saving
    # decide policy: here we save but you can skip by checking column count
    if list(df_ind.columns) == ["t"] or list(df_ind.columns) == ["timestamp"] or (set(df_ind.columns) <= {"t","timestamp"}):
        print("  No mapped tickers with return columns found for this file -> skipping saving.")
        continue

    out_name = fname.replace("panel", "industry_panel")
    out_path = os.path.join(OUTPUT_DIR_INDUSTRY, out_name)
    df_ind.to_csv(out_path, index=False)
    print(f"  Saved {out_path}")
print("Done.")

About to process 228 CSV files (company level -> industry aggregate)
Processing 1000391997969092608_hourly_panel.csv ...
  Saved HOURLY EVENT STUDY PANELS BY INDUSTRY\1000391997969092608_hourly_industry_panel.csv
Processing 1010503423773507584_hourly_panel.csv ...
  Saved HOURLY EVENT STUDY PANELS BY INDUSTRY\1010503423773507584_hourly_industry_panel.csv
Processing 1015586529484443648_hourly_panel.csv ...
  Saved HOURLY EVENT STUDY PANELS BY INDUSTRY\1015586529484443648_hourly_industry_panel.csv
Processing 1019932691339399168_hourly_panel.csv ...
  Saved HOURLY EVENT STUDY PANELS BY INDUSTRY\1019932691339399168_hourly_industry_panel.csv
Processing 1021384752136409088_hourly_panel.csv ...
  Saved HOURLY EVENT STUDY PANELS BY INDUSTRY\1021384752136409088_hourly_industry_panel.csv
Processing 1021917767467982854_hourly_panel.csv ...
  Saved HOURLY EVENT STUDY PANELS BY INDUSTRY\1021917767467982854_hourly_industry_panel.csv
Processing 1023546197129224192_hourly_panel.csv ...
  Saved HOURLY 

### Filtering only mentioned industries return 

In [14]:
# Get files
industry_panel_files = sorted(glob.glob(os.path.join(OUTPUT_DIR_INDUSTRY, "*.csv")))
df_panel_paths = pd.DataFrame({"panel_path": industry_panel_files})

# robust extractor: first try tweet_id_123, then last integer group
def extract_id_from_filename(path):
    base = os.path.splitext(os.path.basename(path))[0]
    m = re.search(r'tweet_id[_\-]?(\d+)', base, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    m2 = re.findall(r'(\d+)', base)
    if m2:
        return int(m2[-1])
    return None

df_panel_paths["id_extracted"] = df_panel_paths["panel_path"].apply(extract_id_from_filename)

# report problematic filenames
bad = df_panel_paths[df_panel_paths["id_extracted"].isna()]
if not bad.empty:
    print("Warning: couldn't parse id for these files:")
    for p in bad["panel_path"].tolist():
        print("  ", p)

# store as pandas nullable integer type (keeps NaNs)
df_panel_paths["id"] = df_panel_paths["id_extracted"].astype("Int64")

# Now ensure df_tweets_filtered.id is same dtype
# (df_tweets_filtered comes from your environment)
df_tweet_id_industries = df_tweets_filtered[["id", "industry"]].copy()
# convert to Int64 to match
df_tweet_id_industries["id"] = df_tweet_id_industries["id"].astype("Int64")

# Merge
df_tweet_id_industries = df_tweet_id_industries.merge(
    df_panel_paths[["panel_path", "id"]],
    on="id",
    how="left",
)

display(df_tweet_id_industries.head())
print("Missing panel_path:", df_tweet_id_industries["panel_path"].isna().sum())

,id,industry,panel_path
0,911287725847908352,Airlines & Travel,HOURLY EVENT STUDY PANELS BY INDUSTRY\91128772...
1,1184147319480041473,Airlines & Travel,HOURLY EVENT STUDY PANELS BY INDUSTRY\11841473...
2,1184631273454817280,Airlines & Travel,HOURLY EVENT STUDY PANELS BY INDUSTRY\11846312...
3,1184987864125321216,Airlines & Travel,HOURLY EVENT STUDY PANELS BY INDUSTRY\11849878...
4,700795170023825408,Technology & Internet,HOURLY EVENT STUDY PANELS BY INDUSTRY\70079517...


Missing panel_path: 0


In [15]:
# Specify output directory and ensure it exists
OUTPUT_DIR_INDUSTRY_SINGLE = "HOURLY EVENT STUDY PANELS BY SINGLE INDUSTRY"
os.makedirs(OUTPUT_DIR_INDUSTRY_SINGLE, exist_ok=True)

# Optional manual overrides for weird industry names -> column names (hourly)
SPECIAL_COLNAME_MAP = {
    "Chemicals": "Chemicals_(Materials_sector)_hourly_return",
    "Semiconductors": "Semiconductors_(Information_Technology_sector)_hourly_return",
}

# Suffix candidates for hourly
RETURN_SUFFIX_CANDIDATES = [
    "_hourly_return",
    "_hour_return",
    "_return",
    "_log_return",
    "_hourly_log_return",
]

# Helper: safe industry prefix
def safe_ind_prefix(industry_str: str) -> str:
    return str(industry_str).replace(" ", "_").replace("/", "_")

# Helper: find return column
def find_industry_return_column(df_columns, industry):
    cols_set = set(df_columns)

    # 1) check explicit override
    if industry in SPECIAL_COLNAME_MAP:
        cand = SPECIAL_COLNAME_MAP[industry]
        if cand in cols_set:
            return cand

    # 2) try safe prefix + suffixes
    safe = safe_ind_prefix(industry)
    for suf in RETURN_SUFFIX_CANDIDATES:
        cand = f"{safe}{suf}"
        if cand in cols_set:
            return cand

    # 3) try raw industry + suffix
    for suf in RETURN_SUFFIX_CANDIDATES:
        cand = f"{industry}{suf}"
        if cand in cols_set:
            return cand

    # 4) fallback heuristic
    tokens = re.findall(r"[A-Za-z0-9]+", industry.lower())
    if tokens:
        for col in df_columns:
            col_l = col.lower()
            if "return" not in col_l:
                continue
            if all(tok in col_l for tok in tokens):
                return col

    return None

# Iterate and create one-file-per-tweet/industry
saved = 0
skipped = 0

for idx, row in df_tweet_id_industries.iterrows():
    tweet_id = row["id"]
    industry = row["industry"]

    panel_path = row.get("panel_path")
    if not panel_path or not os.path.exists(panel_path):
        print(f"[{idx}] Skipping tweet {tweet_id}: panel_path missing or not found.")
        skipped += 1
        continue

    try:
        df = pd.read_csv(panel_path)
    except Exception as e:
        print(f"[{idx}] Skipping tweet {tweet_id} (read error): {e}")
        skipped += 1
        continue

    # require at least one of t/timestamp
    keep_cols = []
    if "t" in df.columns:
        keep_cols.append("t")
    if "timestamp" in df.columns:
        keep_cols.append("timestamp")
    if len(keep_cols) == 0:
        print(f"[{idx}] Skipping tweet {tweet_id}: panel missing both 't' and 'timestamp'.")
        skipped += 1
        continue

    # find return column
    ret_col = find_industry_return_column(df.columns, industry)
    if ret_col is None:
        print(f"[{idx}] Skipping tweet {tweet_id}: no hourly return column found for industry '{industry}'.")
        skipped += 1
        continue

    # build output
    out_df = df[keep_cols + [ret_col]].copy()
    out_df[ret_col] = out_df[ret_col].fillna(0.0)

    safe = safe_ind_prefix(industry)
    out_col_name = f"{safe}_hourly_return"
    out_df = out_df.rename(columns={ret_col: out_col_name})

    # filename logic unchanged
    base_name = os.path.basename(panel_path)
    if "industry" in base_name:
        new_name = base_name.replace("industry", industry[:4].strip(), 1)
    else:
        new_name = f"{tweet_id}_" + base_name

    out_path = Path(OUTPUT_DIR_INDUSTRY_SINGLE) / new_name

    try:
        out_df.to_csv(out_path, index=False)
        saved += 1
    except Exception as e:
        print(f"[{idx}] Failed to write {out_path}: {e}")
        skipped += 1
        continue

print(f"Finished. Saved {saved} files, skipped {skipped}.")

Finished. Saved 238 files, skipped 0.
